# Run summary (last task of the daily job)

Runs even when an earlier task failed (`run_if: ALL_DONE`). Collects row counts per table in Bronze / Silver / Gold, rows added today, and the top 10 of `agg_restaurant_ranking`, then returns them as JSON with `dbutils.notebook.exit`, so a chat session can review a run through the Jobs API (`runs/get-output`) without SQL access.

In [ ]:
# --- Config ---
for layer in ("bronze", "silver", "gold"):
    dbutils.widgets.text(layer, f"prod.{layer}")
LAYERS = {layer: dbutils.widgets.get(layer) for layer in ("bronze", "silver", "gold")}


In [ ]:
# --- Summary ---
import json
from datetime import datetime, timezone, timedelta

today = (datetime.now(timezone.utc) + timedelta(hours=7)).date().isoformat()  # VN date
summary = {"today_vn": today, "tables": {}, "errors": []}

for layer, schema in LAYERS.items():
    for row in spark.sql(f"SHOW TABLES IN {schema}").collect():
        if row.isTemporary:
            continue
        name = f"{schema}.{row.tableName}"
        try:
            cols = set(spark.table(name).columns)
            today_col = next((c for c in ("ingest_date", "snapshot_date", "etl_date") if c in cols), None)
            extra = f", COUNT_IF(CAST({today_col} AS DATE) = DATE'{today}') AS today" if today_col else ""
            r = spark.sql(f"SELECT COUNT(*) AS n{extra} FROM {name}").collect()[0].asDict()
            summary["tables"][f"{layer}.{row.tableName}"] = r
        except Exception as e:
            summary["errors"].append(f"{name}: {str(e)[:200]}")

agg = f"{LAYERS['gold']}.agg_restaurant_ranking"
try:
    df = spark.table(agg)
    order = "legit_score DESC NULLS LAST" if "legit_score" in df.columns else "ranking_score DESC"
    keep = [c for c in ("restaurant_name", "district", "city", "video_count", "credited_views",
                        "google_rating", "legit_score", "legit_label", "ranking_score") if c in df.columns]
    summary["top10"] = [r.asDict() for r in spark.sql(f"SELECT {', '.join(keep)} FROM {agg} ORDER BY {order} LIMIT 10").collect()]
except Exception as e:
    summary["errors"].append(f"{agg}: {str(e)[:200]}")

out = json.dumps(summary, ensure_ascii=False, default=str)
print(out)
dbutils.notebook.exit(out[:900_000])
